# 🔑 PAUTA — ML_U3_Lab01 Redes Neuronales

> **Solo para uso del profesor/ayudante. No distribuir a estudiantes.**
> Versión: 2025-1 | Generada: 2026-05-09


# ML_U3_Lab01 — Redes Neuronales: Implementación y Diagnóstico

**Versión:** 2025-1 | **Modificado:** 2026-05-09
**Dataset:** make_moons + load_breast_cancer (sklearn) | **Duración:** 1 hora
**Modalidad:** Individual o parejas

---

## 📋 Estructura del laboratorio

| Parte | Tema | Tiempo | Audiencia |
|-------|------|--------|-----------|
| Setup | Imports, datos, funciones auxiliares | 5 min | Todos |
| PARTE 1 | Forward pass manual y verificación | 10 min | Todos |
| PARTE 2 | Entrenamiento MLP con sklearn | 20 min | Todos |
| PARTE 3 | Diagnóstico y ajuste de arquitectura | 15 min | Todos |
| ANÁLISIS | Reflexión e interpretación | 10 min | Todos (diferenciado) |

---

## 🎯 Instrucciones por audiencia

| | Pregrado | Doctorado |
|--|----------|-----------|
| Obligatorio | Partes 1, 2, 3 + preguntas azules | Todo lo anterior + TODOs [PhD] + preguntas amarillas |
| Opcional | Bonus azul | Bonus amarillo |
| Entrega | .ipynb ejecutado | .ipynb ejecutado |


## ⚙️ Setup (NO MODIFICAR)

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Funciones auxiliares ──────────────────────────────────────────────────
def plot_decision_boundary(model, X, y, title, ax):
    """Grafica frontera de decisión de un clasificador 2D."""
    h = 0.02
    x0_min, x0_max = X[:,0].min()-0.4, X[:,0].max()+0.4
    x1_min, x1_max = X[:,1].min()-0.4, X[:,1].max()+0.4
    xx, yy = np.meshgrid(np.arange(x0_min, x0_max, h),
                         np.arange(x1_min, x1_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='RdBu', edgecolors='black', s=40, alpha=0.8)
    ax.set_title(title)

# dataset: make_moons + breast_cancer  |  generado sintéticamente / sklearn built-in
X_moons, y_moons = make_moons(n_samples=400, noise=0.25, random_state=RANDOM_STATE)
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

print("✅ Setup completo")
print(f"   Moons: {X_moons.shape}  |  Cancer: {X_cancer.shape}")

---
## PARTE 1 — Ejercicio Sin Computador: Forward Pass Manual (10 min) 🖊️

Antes de ejecutar cualquier celda, **trabaja este ejercicio a mano en papel**.


In [ ]:
# ━━━ PARTE 1: VERIFICACIÓN (ejecutar DESPUÉS de responder a mano) ━━━
W1_p = np.array([[1., 2.], [-1., 0.]])
b1_p = np.array([0., 1.])
W2_p = np.array([0.5, -1.])
b2_p = 0.5
x_p  = np.array([1., -1.])
y_p  = 1.0

z1 = W1_p @ x_p + b1_p
a1 = np.maximum(0, z1)
z2 = W2_p @ a1 + b2_p
y_hat = 1 / (1 + np.exp(-z2))

print("── Forward Pass ──")
print(f"z¹ = {z1}")
print(f"a¹ = {a1}")
print(f"z² = {z2:.4f}")
print(f"ŷ  = {y_hat:.4f}")

# [PhD] Backward pass
loss = -(y_p * np.log(y_hat + 1e-9) + (1-y_p) * np.log(1-y_hat + 1e-9))
delta2 = y_hat - y_p
dW2 = delta2 * a1
delta1 = (W2_p * delta2) * (z1 > 0)
dW1 = np.outer(delta1, x_p)

print("\n── Backward Pass [PhD] ──")
print(f"L    = {loss:.4f}")
print(f"δ²   = {delta2:.4f}")
print(f"∂L/∂W² = {dW2}")
print(f"δ¹   = {delta1}")
print(f"∂L/∂W¹ = {dW1}")

In [ ]:
# 🔍 Tests de sanidad — Parte 1 (NO MODIFICAR)
try:
    assert abs(y_hat - (1/(1+np.exp(-z2)))) < 1e-9, "Error en cálculo de sigmoide"
    assert all(a1 >= 0), "ReLU debe producir solo valores >= 0"
    assert z2 == W2_p @ a1 + b2_p, "Error en z²"
    print("✅ PASS — Forward pass correcto")
except AssertionError as e:
    print(f"❌ FAIL — {e}")

---
## PARTE 2 — Entrenamiento de MLP con scikit-learn (20 min)

Entrena y evalúa un MLP sobre el dataset de **cáncer de mama** (clasificación binaria).
Usa el dataset `load_breast_cancer`: 30 features, 2 clases (maligno/benigno).


In [ ]:
# ━━━ PAUTA PARTE 2: PREPARAR DATOS ━━━
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.2,
    stratify=y_cancer, random_state=RANDOM_STATE
)
print(f"Train: {X_tr.shape}")
print(f"Test:  {X_te.shape}")

In [ ]:
# ━━━ PAUTA PARTE 2: PIPELINE MLP ━━━
pipe_cancer = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(100, 50), activation='relu',
        solver='adam', max_iter=300, random_state=RANDOM_STATE,
        early_stopping=True
    ))
])
pipe_cancer.fit(X_tr, y_tr)
y_pred_cancer = pipe_cancer.predict(X_te)
print(f"Accuracy: {accuracy_score(y_te, y_pred_cancer):.4f}")
print(classification_report(y_te, y_pred_cancer, target_names=cancer.target_names))

In [ ]:
# ━━━ PAUTA PARTE 2: MATRIZ DE CONFUSIÓN ━━━
if pipe_cancer is not None:
    fig, ax = plt.subplots(figsize=(7, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred_cancer,
        display_labels=cancer.target_names,
        cmap='Blues', ax=ax
    )
    plt.title('Matriz de Confusión — MLP en Cáncer de Mama')
    plt.tight_layout()
    plt.show()

In [ ]:
# ━━━ PAUTA PARTE 2: PREPARAR DATOS ━━━
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.2,
    stratify=y_cancer, random_state=RANDOM_STATE
)
print(f"Train: {X_tr.shape}")
print(f"Test:  {X_te.shape}")

---
## PARTE 3 — Diagnóstico y Ajuste de Arquitectura (15 min)

Las curvas de aprendizaje son la herramienta fundamental para diagnosticar
si un modelo tiene **sobreajuste** (overfitting) o **subajuste** (underfitting).


In [ ]:
# ━━━ PAUTA PARTE 3: CURVAS DE APRENDIZAJE ━━━
if pipe_cancer is not None:
    train_sizes, train_scores, val_scores = learning_curve(
        pipe_cancer, X_cancer, y_cancer,
        cv=5, train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='accuracy', n_jobs=-1, random_state=RANDOM_STATE
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    tr_mean = train_scores.mean(axis=1)
    vl_mean = val_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    vl_std  = val_scores.std(axis=1)
    ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.1, color='steelblue')
    ax.fill_between(train_sizes, vl_mean-vl_std, vl_mean+vl_std, alpha=0.1, color='tomato')
    ax.plot(train_sizes, tr_mean, 'o-', color='steelblue', label='Train', lw=2)
    ax.plot(train_sizes, vl_mean, 'o-', color='tomato', label='Validación', lw=2)
    ax.set_xlabel('Tamaño conjunto de entrenamiento'); ax.set_ylabel('Accuracy')
    ax.set_title('Curvas de Aprendizaje — MLP'); ax.legend(); ax.set_ylim(0.7, 1.02)
    plt.tight_layout(); plt.show()
    print(f"Gap train-val final: {tr_mean[-1]-vl_mean[-1]:.4f}")

In [ ]:
# ━━━ PAUTA PARTE 3: COMPARAR ARQUITECTURAS ━━━
X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=RANDOM_STATE
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Arquitectura A: (10,)
pipe_A = Pipeline([('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(10,), activation='relu',
                          max_iter=500, random_state=RANDOM_STATE))])
pipe_A.fit(X_m_tr, y_m_tr)
plot_decision_boundary(pipe_A, X_moons, y_moons,
    f'Arq. A (10,) — acc={accuracy_score(y_m_te, pipe_A.predict(X_m_te)):.3f}', axes[0])

# Arquitectura B: (50, 50)
pipe_B = Pipeline([('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(50,50), activation='relu',
                          max_iter=500, random_state=RANDOM_STATE))])
pipe_B.fit(X_m_tr, y_m_tr)
plot_decision_boundary(pipe_B, X_moons, y_moons,
    f'Arq. B (50,50) — acc={accuracy_score(y_m_te, pipe_B.predict(X_m_te)):.3f}', axes[1])

# [PhD] Arquitectura C: (500, 500) — grande
pipe_C = Pipeline([('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(500,500), activation='relu',
                          max_iter=500, random_state=RANDOM_STATE))])
pipe_C.fit(X_m_tr, y_m_tr)
plot_decision_boundary(pipe_C, X_moons, y_moons,
    f'Arq. C [PhD] (500,500) — acc={accuracy_score(y_m_te, pipe_C.predict(X_m_te)):.3f}', axes[2])

plt.suptitle('Comparación de Arquitecturas — Dataset Moons', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("Nota [PhD]: La frontera de (500,500) es muy irregular → sobreajuste potencial.")
print("Usar validación cruzada y/o regularización (alpha) para controlarlo.")

In [ ]:
# ━━━ PAUTA PARTE 3: CURVAS DE APRENDIZAJE ━━━
if pipe_cancer is not None:
    train_sizes, train_scores, val_scores = learning_curve(
        pipe_cancer, X_cancer, y_cancer,
        cv=5, train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='accuracy', n_jobs=-1, random_state=RANDOM_STATE
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    tr_mean = train_scores.mean(axis=1)
    vl_mean = val_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    vl_std  = val_scores.std(axis=1)
    ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.1, color='steelblue')
    ax.fill_between(train_sizes, vl_mean-vl_std, vl_mean+vl_std, alpha=0.1, color='tomato')
    ax.plot(train_sizes, tr_mean, 'o-', color='steelblue', label='Train', lw=2)
    ax.plot(train_sizes, vl_mean, 'o-', color='tomato', label='Validación', lw=2)
    ax.set_xlabel('Tamaño conjunto de entrenamiento'); ax.set_ylabel('Accuracy')
    ax.set_title('Curvas de Aprendizaje — MLP'); ax.legend(); ax.set_ylim(0.7, 1.02)
    plt.tight_layout(); plt.show()
    print(f"Gap train-val final: {tr_mean[-1]-vl_mean[-1]:.4f}")

---
## Análisis e Interpretación

---
## BONUS (Opcional)

---
## ✅ Checklist de Entrega

### Pregrado
- [ ] PARTE 1: Forward pass a mano respondido y verificado
- [ ] TODO 1: Split implementado
- [ ] TODO 2: Pipeline MLP entrenado con accuracy >= 0.85
- [ ] TODO 3: Matriz de confusión graficada
- [ ] TODO 4: Curvas de aprendizaje generadas
- [ ] TODO 5a y 5b: Arquitecturas comparadas con fronteras de decisión
- [ ] Preguntas 1, 2, 3 respondidas

### Doctorado (adicional)
- [ ] PARTE 1: Backward pass a mano calculado
- [ ] TODO 5 [PhD]: Arquitectura grande comparada y analizada
- [ ] Preguntas 4, 5, 6 respondidas
